In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (average_precision_score, f1_score,
                             balanced_accuracy_score, roc_auc_score)

# ---- Load and align ----
master = pd.read_csv("../data/master_clean.csv")
aa = pd.read_csv("../data/features_aacomp.csv", index_col=0)

keep = master['acc'].isin(aa.index) & master['cluster'].notna()
master = master[keep].reset_index(drop=True)
X = aa.reindex(master['acc']).values
y = master['d2o'].values.astype(int)
groups = master['cluster'].astype(int).values

print(f"Modelling on {len(master)} proteins: "
      f"{y.sum()} positives, {(y==0).sum()} negatives; "
      f"{len(np.unique(groups))} unique clusters")
print(f"Feature shape: {X.shape}")
print(f"Chance AUPRC at this prevalence: {y.mean():.3f}")
print()

# ---- Cluster-aware 5-fold CV ----
cv = GroupKFold(n_splits=5)
fold_scores = []
for fold, (tr, te) in enumerate(cv.split(X, y, groups)):
    assert len(set(groups[tr]) & set(groups[te])) == 0, "Cluster leakage!"

    clf = RandomForestClassifier(
        n_estimators=300, class_weight='balanced',
        n_jobs=-1, random_state=42)
    clf.fit(X[tr], y[tr])
    proba = clf.predict_proba(X[te])[:, 1]
    pred = (proba >= 0.5).astype(int)

    fold_scores.append({
        'fold': fold,
        'n_train': len(tr), 'n_test': len(te),
        'pos_train': int(y[tr].sum()), 'pos_test': int(y[te].sum()),
        'auprc':    average_precision_score(y[te], proba),
        'auroc':    roc_auc_score(y[te], proba),
        'macro_f1': f1_score(y[te], pred, average='macro'),
        'bal_acc':  balanced_accuracy_score(y[te], pred),
    })
    print(f"Fold {fold}: train {len(tr)} ({y[tr].sum()} pos) → "
          f"test {len(te)} ({y[te].sum()} pos) → "
          f"AUPRC={fold_scores[-1]['auprc']:.3f}, "
          f"F1={fold_scores[-1]['macro_f1']:.3f}, "
          f"bal_acc={fold_scores[-1]['bal_acc']:.3f}")

import os
os.makedirs("../results", exist_ok=True)
results = pd.DataFrame(fold_scores)
results.to_csv("../results/thin_slice_aacomp.csv", index=False)

print("\n=== Aggregate (mean ± std across 5 folds) ===")
for col in ['auprc', 'auroc', 'macro_f1', 'bal_acc']:
    print(f"  {col:9s}: {results[col].mean():.3f} ± {results[col].std():.3f}")

Modelling on 1279 proteins: 188 positives, 1091 negatives; 1168 unique clusters
Feature shape: (1279, 21)
Chance AUPRC at this prevalence: 0.147

Fold 0: train 1023 (154 pos) → test 256 (34 pos) → AUPRC=0.130, F1=0.464, bal_acc=0.500
Fold 1: train 1023 (146 pos) → test 256 (42 pos) → AUPRC=0.152, F1=0.455, bal_acc=0.500
Fold 2: train 1023 (154 pos) → test 256 (34 pos) → AUPRC=0.166, F1=0.464, bal_acc=0.500
Fold 3: train 1023 (152 pos) → test 256 (36 pos) → AUPRC=0.169, F1=0.462, bal_acc=0.500
Fold 4: train 1024 (146 pos) → test 255 (42 pos) → AUPRC=0.213, F1=0.455, bal_acc=0.500

=== Aggregate (mean ± std across 5 folds) ===
  auprc    : 0.166 ± 0.031
  auroc    : 0.525 ± 0.054
  macro_f1 : 0.460 ± 0.005
  bal_acc  : 0.500 ± 0.000


In [3]:
esm_data = np.load("../data/features_esm2.npz", allow_pickle=True)
esm_X_all = esm_data['X']
esm_accs = list(esm_data['accs'])
esm_idx = {acc: i for i, acc in enumerate(esm_accs)}

keep2 = master['acc'].isin(esm_accs) & master['cluster'].notna()
master2 = master[keep2].reset_index(drop=True)
X_esm = np.stack([esm_X_all[esm_idx[a]] for a in master2['acc']])
y2 = master2['d2o'].values.astype(int)
groups2 = master2['cluster'].astype(int).values

print(f"ESM-2 modelling on {len(master2)} proteins, feature shape {X_esm.shape}\n")

cv2 = GroupKFold(n_splits=5)
fold_scores2 = []
for fold, (tr, te) in enumerate(cv2.split(X_esm, y2, groups2)):
    assert len(set(groups2[tr]) & set(groups2[te])) == 0
    clf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 n_jobs=-1, random_state=42)
    clf.fit(X_esm[tr], y2[tr])
    proba = clf.predict_proba(X_esm[te])[:, 1]
    pred = (proba >= 0.5).astype(int)
    fold_scores2.append({
        'fold': fold,
        'auprc':    average_precision_score(y2[te], proba),
        'auroc':    roc_auc_score(y2[te], proba),
        'macro_f1': f1_score(y2[te], pred, average='macro'),
        'bal_acc':  balanced_accuracy_score(y2[te], pred),
    })
    print(f"Fold {fold}: AUPRC={fold_scores2[-1]['auprc']:.3f}, "
          f"AUROC={fold_scores2[-1]['auroc']:.3f}, "
          f"F1={fold_scores2[-1]['macro_f1']:.3f}, "
          f"bal_acc={fold_scores2[-1]['bal_acc']:.3f}")

results2 = pd.DataFrame(fold_scores2)
results2.to_csv("../results/thin_slice_esm2.csv", index=False)
print("\n=== ESM-2 aggregate ===")
for col in ['auprc', 'auroc', 'macro_f1', 'bal_acc']:
    print(f"  {col:9s}: {results2[col].mean():.3f} ± {results2[col].std():.3f}")

ESM-2 modelling on 1279 proteins, feature shape (1279, 1280)

Fold 0: AUPRC=0.136, AUROC=0.497, F1=0.464, bal_acc=0.500
Fold 1: AUPRC=0.204, AUROC=0.505, F1=0.455, bal_acc=0.500
Fold 2: AUPRC=0.157, AUROC=0.550, F1=0.464, bal_acc=0.500
Fold 3: AUPRC=0.217, AUROC=0.586, F1=0.461, bal_acc=0.498
Fold 4: AUPRC=0.253, AUROC=0.679, F1=0.455, bal_acc=0.500

=== ESM-2 aggregate ===
  auprc    : 0.193 ± 0.047
  auroc    : 0.563 ± 0.074
  macro_f1 : 0.460 ± 0.005
  bal_acc  : 0.500 ± 0.001


In [4]:
# === Diagnostic 1: Can RF even fit (train-set check)? ===
clf_overfit = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                      n_jobs=-1, random_state=42)
clf_overfit.fit(X_esm, y2)
proba_train = clf_overfit.predict_proba(X_esm)[:, 1]
print("Train-set AUPRC (RF self-fit):", round(average_precision_score(y2, proba_train), 3))
print("Train-set AUROC (RF self-fit):", round(roc_auc_score(y2, proba_train), 3))
print()

# === Diagnostic 2: Logistic Regression with L2 ===
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

fold_scores_lr = []
for fold, (tr, te) in enumerate(cv2.split(X_esm, y2, groups2)):
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_esm[tr])
    X_te = sc.transform(X_esm[te])
    clf = LogisticRegression(C=1.0, class_weight='balanced',
                             max_iter=2000, random_state=42, n_jobs=-1)
    clf.fit(X_tr, y2[tr])
    proba = clf.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    fold_scores_lr.append({
        'fold': fold,
        'auprc': average_precision_score(y2[te], proba),
        'auroc': roc_auc_score(y2[te], proba),
        'macro_f1': f1_score(y2[te], pred, average='macro'),
        'bal_acc': balanced_accuracy_score(y2[te], pred),
    })
    print(f"LR Fold {fold}: AUPRC={fold_scores_lr[-1]['auprc']:.3f}, "
          f"AUROC={fold_scores_lr[-1]['auroc']:.3f}, "
          f"F1={fold_scores_lr[-1]['macro_f1']:.3f}, "
          f"bal_acc={fold_scores_lr[-1]['bal_acc']:.3f}")

results_lr = pd.DataFrame(fold_scores_lr)
print("\n=== Logistic Regression (ESM-2) aggregate ===")
for col in ['auprc', 'auroc', 'macro_f1', 'bal_acc']:
    print(f"  {col:9s}: {results_lr[col].mean():.3f} ± {results_lr[col].std():.3f}")

Train-set AUPRC (RF self-fit): 1.0
Train-set AUROC (RF self-fit): 1.0

LR Fold 0: AUPRC=0.125, AUROC=0.461, F1=0.462, bal_acc=0.462
LR Fold 1: AUPRC=0.216, AUROC=0.560, F1=0.536, bal_acc=0.540
LR Fold 2: AUPRC=0.193, AUROC=0.515, F1=0.508, bal_acc=0.508
LR Fold 3: AUPRC=0.203, AUROC=0.619, F1=0.558, bal_acc=0.585
LR Fold 4: AUPRC=0.244, AUROC=0.645, F1=0.554, bal_acc=0.561

=== Logistic Regression (ESM-2) aggregate ===
  auprc    : 0.197 ± 0.044
  auroc    : 0.560 ± 0.075
  macro_f1 : 0.524 ± 0.040
  bal_acc  : 0.531 ± 0.048


/opt/miniconda3/envs/biol466/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/miniconda3/envs/biol466/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/miniconda3/envs/biol466/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/miniconda3/envs/biol466/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. 

In [5]:
from sklearn.decomposition import PCA

fold_scores_pca = []
for fold, (tr, te) in enumerate(cv2.split(X_esm, y2, groups2)):
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_esm[tr])
    X_te_s = sc.transform(X_esm[te])
    pca = PCA(n_components=64, random_state=42)
    X_tr_p = pca.fit_transform(X_tr_s)
    X_te_p = pca.transform(X_te_s)

    clf = RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                 n_jobs=-1, random_state=42,
                                 min_samples_leaf=5)
    clf.fit(X_tr_p, y2[tr])
    proba = clf.predict_proba(X_te_p)[:, 1]
    pred = (proba >= 0.5).astype(int)
    fold_scores_pca.append({
        'auprc': average_precision_score(y2[te], proba),
        'auroc': roc_auc_score(y2[te], proba),
        'macro_f1': f1_score(y2[te], pred, average='macro'),
        'bal_acc': balanced_accuracy_score(y2[te], pred),
    })
    print(f"PCA64+RF Fold {fold}: AUPRC={fold_scores_pca[-1]['auprc']:.3f}, "
          f"AUROC={fold_scores_pca[-1]['auroc']:.3f}, "
          f"F1={fold_scores_pca[-1]['macro_f1']:.3f}, "
          f"bal_acc={fold_scores_pca[-1]['bal_acc']:.3f}")

results_pca = pd.DataFrame(fold_scores_pca)
print("\n=== ESM-2 → PCA(64) → RF aggregate ===")
for col in ['auprc', 'auroc', 'macro_f1', 'bal_acc']:
    print(f"  {col:9s}: {results_pca[col].mean():.3f} ± {results_pca[col].std():.3f}")

PCA64+RF Fold 0: AUPRC=0.156, AUROC=0.485, F1=0.464, bal_acc=0.500
PCA64+RF Fold 1: AUPRC=0.193, AUROC=0.478, F1=0.455, bal_acc=0.500
PCA64+RF Fold 2: AUPRC=0.151, AUROC=0.499, F1=0.464, bal_acc=0.500
PCA64+RF Fold 3: AUPRC=0.203, AUROC=0.641, F1=0.462, bal_acc=0.500
PCA64+RF Fold 4: AUPRC=0.249, AUROC=0.638, F1=0.455, bal_acc=0.500

=== ESM-2 → PCA(64) → RF aggregate ===
  auprc    : 0.190 ± 0.040
  auroc    : 0.548 ± 0.084
  macro_f1 : 0.460 ± 0.005
  bal_acc  : 0.500 ± 0.000
